
# ARC-AGI-3 Duck v12 — Qwen/Qwen3.8-27B-FP8 + ADLDB-DWE — ALL INPUTS FIXED

**Control seed:** `20260819`

This build is hard-locked to **Qwen/Qwen3.8-27B-FP8**. It validates the mounted
Qwen3.8 snapshot before server startup and validates `/v1/models` again after
vLLM starts. A Qwen3.6 server causes an immediate abort before gameplay.

The full per-move exploit trace remains enabled through `DWE PRE` and
`DWE POST` for every hooked real environment action.


In [ ]:
# === KAGGLE INPUT ATTACHMENT PREFLIGHT ===
from pathlib import Path

_required_input_checks = {
    "ARC competition": [
        Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
    ],
    "TAAF source": [
        Path("/kaggle/input/taaf-kaggle-source-share"),
        Path("/kaggle/input/datasets/jeroencottaar/taaf-kaggle-source-share"),
    ],
    "vLLM wheelhouse": [
        Path("/kaggle/input/arc3-vllm-h100-wheelhouse-v3"),
        Path("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"),
    ],
    "Qwen3.8 model": [
        Path("/kaggle/input/qwen3-8-27b-fp8-hf-017b9c7a"),
        Path("/kaggle/input/datasets/driessmit1/qwen3-8-27b-fp8-hf-017b9c7a"),
    ],
}

print("=" * 86)
print("KAGGLE INPUT PREFLIGHT")
_missing = []
for label, candidates in _required_input_checks.items():
    found = next((p for p in candidates if p.exists()), None)
    if found is None:
        print(f"MISSING  {label}")
        _missing.append(label)
    else:
        print(f"FOUND    {label}: {found}")

# Also permit the notebook's later content-based Qwen3.8 resolver to discover
# a different Kaggle mount layout / compatible fallback snapshot.
if "Qwen3.8 model" in _missing:
    try:
        any_qwen38 = any(
            ("qwen3-8" in str(p).lower() or "qwen38" in str(p).lower())
            for p in Path("/kaggle/input").rglob("config.json")
        )
    except OSError:
        any_qwen38 = False
    if any_qwen38:
        print("FOUND    Qwen3.8-compatible model via content scan")
        _missing.remove("Qwen3.8 model")

print("=" * 86)
if _missing:
    raise RuntimeError(
        "KAGGLE INPUTS NOT ATTACHED: "
        + ", ".join(_missing)
        + ". For the complete four-input run, push the supplied folder using "
          "`kaggle kernels push -p <folder>` instead of uploading only the .ipynb."
    )


In [ ]:
import json
import random
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Match the public Duck-v12 control seed and propagate it through every layer
# that can be controlled before the harness/server is imported.
CONTROL_SEED = 20260819
ANALYZER_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
os.environ["PYTHONHASHSEED"] = str(CONTROL_SEED)
os.environ["ARC3_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["TAAF_SEED"] = str(CONTROL_SEED)
os.environ["VLLM_SEED"] = str(CONTROL_SEED)

# Documented Duck harness perception control.
os.environ["ARC3_FRAME_MODE"] = "full"
# Explicitly keep the state-graph experiment off; public ablations found it costly.
os.environ["ARC3_STATE_GRAPH"] = "off"
# These are intentionally descriptive aliases as well; harmless when a source
# bundle does not consume them and useful when a later bundle does.
os.environ["TAAF_FRAME_MODE"] = "full"
os.environ["ARC3_NO_IMPACT_DWE"] = "1"

random.seed(CONTROL_SEED)


# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Pin the analyzer to the best-known GPU-backed runner configuration.
os.environ["INFERENCE_ANALYZER_MODEL"] = "Qwen/Qwen3.8-27B-FP8"
os.environ["LOCAL_ANALYZER_MODEL_ID"] = "Qwen/Qwen3.8-27B-FP8"
os.environ.setdefault("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
os.environ.setdefault("TAAF_MAX_OUTPUT_TOKENS", "8192")
os.environ.setdefault("TAAF_TOOL_STEPS", "8")
os.environ.setdefault("TAAF_TEMPERATURE", "0.6")
os.environ.setdefault("TAAF_TOP_P", "0.95")
os.environ["TAAF_CONTEXT_WINDOW"] = "32768"
os.environ.setdefault("MULTIMODAL_CONTEXT", "1")

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).


In [ ]:
# === REQUIRED INPUT BOOTSTRAP — RUNS BEFORE ANY INSTALL ===
# Resolves every external input used by the scored notebook and publishes
# canonical paths for all later cells.
from pathlib import Path
import json
import os
import sys

REQUIRED_COMPETITION = "arc-prize-2026-arc-agi-3"
REQUIRED_DATASET_SOURCES = [
    "jeroencottaar/taaf-kaggle-source-share",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
    "driessmit1/qwen3-8-27b-fp8-hf-017b9c7a",
]
DATASET_SOURCES = list(REQUIRED_DATASET_SOURCES)
FALLBACK_MODEL_DATASET_SOURCE = "mustangliu/qwen38-27b-fp8-hf-snapshot"
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "datasets" / owner / slug,
    ]


def _competition_mount_candidates(slug: str) -> list[Path]:
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "competitions" / slug,
    ]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((p for p in candidates if p.exists()), None)


def _find_named(root: Path, name: str) -> Path | None:
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require(path: Path | None, label: str) -> Path:
    if path is None or not path.exists():
        raise FileNotFoundError(
            f"REQUIRED INPUT MISSING: {label}. "
            "Attach it in Kaggle or push this notebook with the included "
            "kernel-metadata.json."
        )
    return path.resolve()


def _resolve_dataset(ref: str, marker: str | None = None) -> Path:
    candidates = _dataset_mount_candidates(ref)
    mounted = _first_existing(candidates)
    if mounted is not None:
        if marker is None or _find_named(mounted, marker) is not None:
            return mounted.resolve()

    if marker is not None and KAGGLE_INPUT_ROOT.exists():
        hit = _find_named(KAGGLE_INPUT_ROOT, marker)
        if hit is not None:
            return hit.parent.resolve()

    expected = " OR ".join(str(p) for p in candidates)
    raise FileNotFoundError(
        f"REQUIRED DATASET MISSING: {ref}. Expected mount: {expected}"
    )


# 1) ARC-AGI-3 competition source.
ARC_COMPETITION_ROOT = _first_existing(
    _competition_mount_candidates(REQUIRED_COMPETITION)
)
if ARC_COMPETITION_ROOT is None and KAGGLE_INPUT_ROOT.exists():
    # Support mount-layout changes by locating the competition's unique wheel dir.
    hit = _find_named(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if hit is not None and hit.is_dir():
        ARC_COMPETITION_ROOT = hit.parent

ARC_COMPETITION_ROOT = _require(
    ARC_COMPETITION_ROOT,
    f"competition:{REQUIRED_COMPETITION}",
)

ARC_WHEELS_DIR = ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
if not ARC_WHEELS_DIR.is_dir():
    hit = _find_named(ARC_COMPETITION_ROOT, "arc_agi_3_wheels")
    ARC_WHEELS_DIR = _require(
        hit if hit is not None and hit.is_dir() else None,
        "ARC competition wheel directory: arc_agi_3_wheels",
    )
else:
    ARC_WHEELS_DIR = ARC_WHEELS_DIR.resolve()

ARC_ENVIRONMENTS_DIR = ARC_COMPETITION_ROOT / "environment_files"
if not TRUE_SUBMISSION:
    ARC_ENVIRONMENTS_DIR = _require(
        ARC_ENVIRONMENTS_DIR if ARC_ENVIRONMENTS_DIR.is_dir() else None,
        "ARC offline environment_files",
    )
else:
    ARC_ENVIRONMENTS_DIR = ARC_ENVIRONMENTS_DIR.resolve()

# 2) Duck/TAAF source bundle.
TAAF_BUNDLE_MOUNT = _resolve_dataset(
    REQUIRED_DATASET_SOURCES[0],
    DATASET_BUNDLE_MARKER,
)
bundle_marker = _find_named(TAAF_BUNDLE_MOUNT, DATASET_BUNDLE_MARKER)
if bundle_marker is None:
    raise FileNotFoundError(
        f"TAAF bundle marker {DATASET_BUNDLE_MARKER!r} not found under "
        f"{TAAF_BUNDLE_MOUNT}"
    )
BUNDLE_DIR = bundle_marker.parent.resolve()

for required_name in ("src", "setup_commands.json", "teardown_commands.json"):
    _require(BUNDLE_DIR / required_name, f"TAAF bundle component:{required_name}")

# 3) Offline vLLM wheelhouse.
VLLM_WHEELHOUSE_MOUNT = _resolve_dataset(
    REQUIRED_DATASET_SOURCES[1],
    "requirements.lock",
)
wheel_lock = _find_named(VLLM_WHEELHOUSE_MOUNT, "requirements.lock")
if wheel_lock is None:
    raise FileNotFoundError(
        f"vLLM wheelhouse requirements.lock not found under "
        f"{VLLM_WHEELHOUSE_MOUNT}"
    )
VLLM_WHEELHOUSE_DIR = wheel_lock.parent.resolve()

# 4) Qwen3.8-27B-FP8 snapshot.
# Prefer the Driessmit snapshot used by ARC-AGI-3 notebooks. If a user manually
# attaches the Mustang snapshot instead, accept it without changing notebook code.
def _resolve_qwen38_model() -> tuple[str, Path]:
    candidates = [
        REQUIRED_DATASET_SOURCES[2],
        FALLBACK_MODEL_DATASET_SOURCE,
    ]
    errors = []
    for ref in candidates:
        try:
            mount = _first_existing(_dataset_mount_candidates(ref))
            if mount is None:
                raise FileNotFoundError(
                    "dataset mount not present: "
                    + " OR ".join(str(p) for p in _dataset_mount_candidates(ref))
                )
            config = _find_named(mount, "config.json")
            if config is None:
                raise FileNotFoundError(f"config.json not found under {mount}")
            model_dir = config.parent.resolve()
            index = model_dir / "model.safetensors.index.json"
            single = model_dir / "model.safetensors"
            shards = list(model_dir.glob("*.safetensors"))
            if not index.exists() and not single.exists() and not shards:
                raise FileNotFoundError(
                    f"No safetensors weights found in {model_dir}"
                )
            return ref, model_dir
        except Exception as exc:
            errors.append(f"{ref}: {exc}")
    raise FileNotFoundError(
        "No usable Qwen3.8-27B-FP8 dataset was mounted. Tried:\n- "
        + "\n- ".join(errors)
    )


RESOLVED_MODEL_DATASET_SOURCE, QWEN_MODEL_DIR = _resolve_qwen38_model()
weight_index = QWEN_MODEL_DIR / "model.safetensors.index.json"
single_weight = QWEN_MODEL_DIR / "model.safetensors"
weight_shards = list(QWEN_MODEL_DIR.glob("*.safetensors"))

# Validate that the model config is actually a Qwen 3.8-family snapshot when
# identifying metadata is present. We warn instead of guessing on custom configs.
_model_cfg = json.loads((QWEN_MODEL_DIR / "config.json").read_text(encoding="utf-8"))
_model_text = json.dumps(_model_cfg, sort_keys=True).lower()
_model_path_text = str(QWEN_MODEL_DIR).lower()
_resolved_ref_text = str(RESOLVED_MODEL_DATASET_SOURCE).lower()

if "qwen" not in _model_text:
    raise RuntimeError("Resolved model config is not a Qwen-family model.")

if not (
    "qwen3-8" in _resolved_ref_text
    or "qwen38" in _resolved_ref_text
    or "qwen3.8" in _model_path_text
    or "qwen3-8" in _model_path_text
    or "qwen38" in _model_path_text
):
    raise RuntimeError(
        "MODEL IDENTITY CHECK FAILED: expected a Qwen3.8 snapshot, "
        f"resolved={RESOLVED_MODEL_DATASET_SOURCE} path={QWEN_MODEL_DIR}"
    )

EXPECTED_SERVED_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
print(
    "MODEL IDENTITY CHECK PASS "
    f"dataset={RESOLVED_MODEL_DATASET_SOURCE} "
    f"path={QWEN_MODEL_DIR} "
    f"expected_served_id={EXPECTED_SERVED_MODEL_ID}",
    flush=True,
)

# Canonical input map for setup scripts and later runtime cells.
# LEGACY_MODEL_DATASET_SOURCE is a compatibility key only: older TAAF source
# bundles may look up that exact key, but it resolves to the Qwen3.8 directory.
LEGACY_MODEL_DATASET_SOURCE = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"
kaggle_input_paths: dict[str, str] = {
    REQUIRED_DATASET_SOURCES[0]: str(BUNDLE_DIR),
    REQUIRED_DATASET_SOURCES[1]: str(VLLM_WHEELHOUSE_DIR),
    REQUIRED_DATASET_SOURCES[2]: str(QWEN_MODEL_DIR),
    RESOLVED_MODEL_DATASET_SOURCE: str(QWEN_MODEL_DIR),
    LEGACY_MODEL_DATASET_SOURCE: str(QWEN_MODEL_DIR),
    f"competition:{REQUIRED_COMPETITION}": str(ARC_COMPETITION_ROOT),
}

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "ARC_AGI3_COMPETITION_ROOT": str(ARC_COMPETITION_ROOT),
    "ARC_AGI3_WHEELS_DIR": str(ARC_WHEELS_DIR),
    "ARC_AGI3_ENVIRONMENTS_DIR": str(ARC_ENVIRONMENTS_DIR),
    "TAAF_VLLM_WHEELHOUSE": str(VLLM_WHEELHOUSE_DIR),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "LOCAL_ANALYZER_MODEL_PATH": str(QWEN_MODEL_DIR),
    "ARC3_CONTROL_SEED": str(CONTROL_SEED),
    "ARC3_FRAME_MODE": "full",
    "ARC3_STATE_GRAPH": "off",
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(
    json.dumps(setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

INPUT_MANIFEST = {
    "competition": {
        "source": REQUIRED_COMPETITION,
        "root": str(ARC_COMPETITION_ROOT),
        "wheels": str(ARC_WHEELS_DIR),
        "environment_files": str(ARC_ENVIRONMENTS_DIR),
    },
    "datasets": {
        REQUIRED_DATASET_SOURCES[0]: str(BUNDLE_DIR),
        REQUIRED_DATASET_SOURCES[1]: str(VLLM_WHEELHOUSE_DIR),
        RESOLVED_MODEL_DATASET_SOURCE: str(QWEN_MODEL_DIR),
    },
    "model": {
        "requested": REQUIRED_DATASET_SOURCES[2],
        "resolved": RESOLVED_MODEL_DATASET_SOURCE,
        "path": str(QWEN_MODEL_DIR),
        "legacy_source_alias": LEGACY_MODEL_DATASET_SOURCE,
    },
    "true_submission": bool(TRUE_SUBMISSION),
}

(WORKING_DIR / "adldb_input_manifest.json").write_text(
    json.dumps(INPUT_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=" * 88)
print("ADLDB REQUIRED INPUTS — ALL RESOLVED")
print(f"competition root : {ARC_COMPETITION_ROOT}")
print(f"ARC wheels       : {ARC_WHEELS_DIR}")
print(f"environment files: {ARC_ENVIRONMENTS_DIR}")
print(f"TAAF source      : {BUNDLE_DIR}")
print(f"vLLM wheelhouse  : {VLLM_WHEELHOUSE_DIR}")
print(f"Qwen3.8 dataset : {RESOLVED_MODEL_DATASET_SOURCE}")
print(f"Qwen3.8 model   : {QWEN_MODEL_DIR}")
print(f"control seed    : {CONTROL_SEED}")
print(f"frame mode      : {os.environ.get('ARC3_FRAME_MODE')}")
print(f"model shards     : {len(weight_shards)}")
print("=" * 88)


In [ ]:
# === INSTALL ARC RUNTIME FROM THE RESOLVED COMPETITION INPUT ===
# No internet is required.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(ARC_WHEELS_DIR),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
print(f"taaf.kaggle: arc-agi installed from {ARC_WHEELS_DIR}")


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.


In [ ]:
# === PUBLISH RESOLVED KAGGLE INPUTS TO TAAF ===
# Input resolution already ran before installation; this cell exposes those
# validated mounts using the interface expected by the source bundle.
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env.update(
    {
        "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
        "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
        "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    }
)
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(
    json.dumps(setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.


In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()

def _rewrite_setup_command(command: str) -> str:
    # Compatibility bridge for source bundles authored against the previous
    # Qwen3.6 dataset. The command still executes the original setup logic, but
    # any old model source/path is redirected to the resolved Qwen3.8 snapshot.
    replacements = {
        "/kaggle/input/vrfai-qwen3-6-27b-fp8-hf-snapshot": str(QWEN_MODEL_DIR),
        "/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": str(QWEN_MODEL_DIR),
        "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": RESOLVED_MODEL_DATASET_SOURCE,
        "vrfai/Qwen3.6-27B-FP8": "Qwen/Qwen3.8-27B-FP8",
    }
    rewritten = str(command)
    for old, new in replacements.items():
        rewritten = rewritten.replace(old, new)
    return rewritten

for raw_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    command = _rewrite_setup_command(raw_command)
    if command != raw_command:
        print(
            "taaf.kaggle: setup command model path upgraded "
            f"from legacy Qwen3.6 reference to {RESOLVED_MODEL_DATASET_SOURCE}",
            flush=True,
        )
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')

# Re-apply deterministic seeds after setup commands import/install numerical
# libraries. CUDA determinism is best-effort because vLLM may use kernels whose
# execution ordering is not bitwise deterministic.
random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception as _exc:
    print(f"taaf.kaggle: numpy seed warning: {_exc}", flush=True)

try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception as _exc:
    print(f"taaf.kaggle: torch seed warning: {_exc}", flush=True)

# Adopt the actual model ID exposed by the local vLLM server instead of assuming
# a served-model-name. This prevents dataset-directory names from breaking calls.
def _resolve_served_model_id(base_url: str, timeout_s: int = 180) -> str:
    from urllib.request import urlopen
    endpoint = base_url.rstrip("/") + "/models"
    deadline = time.monotonic() + timeout_s
    last_error = None
    while time.monotonic() < deadline:
        try:
            with urlopen(endpoint, timeout=10) as response:
                payload = json.loads(response.read().decode("utf-8"))
            models = payload.get("data", [])
            if models and models[0].get("id"):
                return str(models[0]["id"])
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(
        f"Local analyzer server did not expose a model at {endpoint}: {last_error}"
    )

_served_model_id = _resolve_served_model_id(
    os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
)

if _served_model_id != "Qwen/Qwen3.8-27B-FP8":
    raise RuntimeError(
        "WRONG MODEL SERVED: "
        f"expected=Qwen/Qwen3.8-27B-FP8 actual={_served_model_id}. "
        "Aborting before ARC gameplay."
    )

os.environ["INFERENCE_ANALYZER_MODEL"] = _served_model_id
os.environ["LOCAL_ANALYZER_MODEL_ID"] = _served_model_id
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(
    {
        "INFERENCE_ANALYZER_MODEL": _served_model_id,
        "LOCAL_ANALYZER_MODEL_ID": _served_model_id,
        "ARC3_CONTROL_SEED": str(CONTROL_SEED),
        "ARC3_FRAME_MODE": "full",
        "ARC3_STATE_GRAPH": "off",
    }
)
SETUP_ENV_PATH.write_text(
    json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(
    "CONTROL STACK READY "
    f"served_model={_served_model_id} seed={CONTROL_SEED} "
    f"frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"context={os.environ.get('TAAF_CONTEXT_WINDOW')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')}",
    flush=True,
)


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. ADLDB + Difference-Weighted Exploitation configuration

Every discovered game is still run once. `LS20_MAX_MOVES` remains the absolute safety ceiling, while DWE computes a **live per-game budget** from current-game evidence. A successful transition gets a protected exploit window; repeated no-progress, stalls, and loops reduce strategy weight and can trigger policy change or stop-loss.


In [ ]:
# === ADLDB / DIFFERENCE-WEIGHTED EXPLOITATION CONFIGURATION ===
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4
TARGET_SCORE_GAMES = 10
TARGET_MIN_GAME_SCORE = 0.40

# Absolute safety ceiling. DWE normally allocates a lower live budget and raises it
# only when current-game evidence justifies more computation.
LS20_MAX_MOVES = 309
MAX_STALL_ACTIONS = 12
MAX_NO_PROGRESS_ACTIONS = 24
MIN_OBSERVATION_ACTIONS = 8
SUCCESS_PROTECT_ACTIONS = 18
DWE_STRICT_LOG_COVERAGE = True
NO_IMPACT_STREAK_FOR_POLICY_CHANGE = 3
NO_IMPACT_STREAK_FOR_STOP = 8

# Difference-Weighted Exploitation signals. Positive terms reward causal evidence;
# negative terms penalize wasted trajectories. These are current-game-only signals.
EXPLOIT_WEIGHTS = {
    "score": 1.50,
    "level_complete": 4.00,
    "progress_velocity": 2.25,
    "novel_state": 1.00,
    "causal_confidence": 1.75,
    "target_proximity": 2.50,
    "stall": -2.00,
    "repeat_loop": -3.50,
    "no_progress": -2.75,
    "no_impact": -4.25,
    "terminal_loss": -4.00,
}

# Game weight answers: "is this game worth more global computation?"
# Strategy weight answers: "is the current local behavior worth repeating?"
GAME_WEIGHT_DECAY = 0.94
STRATEGY_WEIGHT_DECAY = 0.88
GAME_WEIGHT_LIMIT = 12.0
STRATEGY_WEIGHT_LIMIT = 12.0

# Combined-weight -> live action budget. This is a soft budget; the hard ceiling
# remains LS20_MAX_MOVES and successful trajectories receive a protected window.
DWE_BUDGET_TIERS = (
    (6.0, 309),   # HARD_EXPLOIT
    (3.0, 260),   # EXPLOIT
    (1.0, 210),   # CAUTIOUS_EXPLOIT
    (-1.0, 160),  # BALANCED
    (-3.0, 120),  # EXPLORE / policy transition
    (-999.0, 84), # probable stop-loss trajectory
)

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_actions_per_game = LS20_MAX_MOVES

# Optional grafts stay within the same current game and current run. A larger
# working context is retained; compact DWE/ADL state prevents raw-history growth.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert bm.solver.max_actions_per_game == LS20_MAX_MOVES
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"hard_action_ceiling={LS20_MAX_MOVES} "
    f"stall={MAX_STALL_ACTIONS} "
    f"no_progress={MAX_NO_PROGRESS_ACTIONS} "
    f"success_protect={SUCCESS_PROTECT_ACTIONS} "
    f"context={_graft_flags['context_window']} "
    f"seed={CONTROL_SEED} frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')} "
    f"source_per_game_budget={_original_game_budget}",
    flush=True,
)
print("DWE WEIGHTS:", json.dumps(EXPLOIT_WEIGHTS, sort_keys=True), flush=True)


## 7. Closed-loop ADL + Difference-Weighted Exploitation

The model still performs the two-plan `EXPLOIT` vs `EXPLORE` comparison before each action and a post-move ADL update afterward. In addition, this cell installs a **deterministic runtime DWE auditor** at the real `GameAPI` action boundary.

For every committed action the notebook prints:

- `DWE PRE`: prior game/strategy weights, current decision, live budget, stall/no-progress counters.
- `DWE POST`: before/after score and levels, reward, board/state-change evidence, novelty/loop signals, each weighted term, new weights, and the next allocator decision.

The same records are written to `/kaggle/working/dwe_move_events.jsonl`.


In [ ]:
# === CLOSED-LOOP ADL + DIFFERENCE-WEIGHTED EXPLOITATION ===
import asyncio
import contextvars
import hashlib
import inspect
import json
import math
import os
import sys
import threading
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DWE_MOVE_LOG = WORKING_DIR / "dwe_move_events.jsonl"
DWE_SUMMARY_LOG = WORKING_DIR / "dwe_game_summaries.jsonl"

for _path in (DUAL_PATH_POLICY_LOG, POST_MOVE_ADL_LOG, DWE_MOVE_LOG, DWE_SUMMARY_LOG):
    try:
        _path.unlink(missing_ok=True)
    except Exception:
        pass

CLOSED_LOOP_ADL_INSTRUCTION = r"""
ADLDB CLOSED-LOOP DUAL-PATH + DIFFERENCE-WEIGHTED EXPLOITATION POLICY

You have exactly ONE real environment trajectory for the current game. Never
fork, clone, reset for speculation, or use another game's state. Use only
observations/actions/rewards/transitions learned in THIS game in THIS run.

BEFORE EVERY REAL ACTION
1. Construct exactly two legal candidate actions from the same current state:
   A = EXPLOIT: shortest move supported by confirmed causal evidence.
   B = EXPLORE: highest-information legal move not already exhausted.
2. Compare legality, predicted progress, predicted frame/state change,
   information gain, loop risk, action cost, and current-game consistency.
3. Maintain two current-game-only values:
   GAME_EXPLOIT_WEIGHT: whether this GAME deserves more computation.
   STRATEGY_EXPLOIT_WEIGHT: whether the CURRENT STRATEGY deserves repetition.
4. Print/record in your reasoning trace before the tool call:

DWE_PRE_DECISION:
STEP=<integer>
GAME_WEIGHT=<number>
STRATEGY_WEIGHT=<number>
A_ACTION=<candidate A>
B_ACTION=<candidate B>
SELECT=<A or B>
MODE=<HARD_EXPLOIT|EXPLOIT|CAUTIOUS_EXPLOIT|BALANCED|EXPLORE|CHANGE_POLICY>
WHY=<current-game evidence only>

Then issue exactly ONE real environment action.

IMMEDIATELY AFTER EVERY REAL ACTION
Compare pre-state, prediction, action and actual returned state. Record:

POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<number or unknown>
LEVEL_DELTA=<number or unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game lesson>
NEXT_BIAS=<exploit/explore/change_policy/stop_loss/neutral>

Perception/control principles:
- use the full current frame; treat animation/change as evidence, not decoration;
- optimize level depth and verified score progress;
- a visual change confined to a deterministic HUD/moves band is NO_IMPACT;
- ACTION7 is legal when exposed by the environment;

DWE exploitation principles:
- verified level completion is the strongest positive signal;
- positive score/reward/progress increases both game and strategy value;
- novel useful transitions increase information value;
- repeated unchanged states, loops and no-progress streaks reduce strategy value;
- a promising game with a weak strategy means CHANGE_POLICY, not immediate abandon;
- sustained low game value + low strategy value + no progress means STOP_LOSS;
- after verified success, exploit the causal pattern for a protected window;
- never let one lucky early transition permanently monopolize the budget.

The runtime independently audits these decisions and prints DWE PRE / DWE POST
for every actual environment action. Your next action must use the latest
current-game ADL evidence rather than historical/cross-game knowledge.
""".strip()


class ClosedLoopADLToolAgent(ToolAgent):
    """Duck ToolAgent with dual-path ADL and explicit DWE reasoning requirements."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "Qwen/Qwen3.8-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    kwargs = {
        "model": model,
        "timeout": bm.solver.analyzer_timeout,
        "save_request_logs": bm.solver.save_request_logs,
        "base_url": base_url,
        "provider": "vllm",
    }
    # Preserve compatibility with multiple ToolAgent versions while passing the
    # control seed at request level whenever the installed implementation
    # explicitly supports a seed-bearing argument.
    try:
        base_params = inspect.signature(ToolAgent.__init__).parameters
    except Exception:
        base_params = {}
    if "seed" in base_params:
        kwargs["seed"] = CONTROL_SEED
    elif "request_kwargs" in base_params:
        kwargs["request_kwargs"] = {"seed": CONTROL_SEED}
    elif "model_kwargs" in base_params:
        kwargs["model_kwargs"] = {"seed": CONTROL_SEED}
    return ClosedLoopADLToolAgent(**kwargs)


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory


def _clip(value, low, high):
    return max(low, min(high, float(value)))


def _num(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        x = float(value)
        if math.isfinite(x):
            return x
    except Exception:
        pass
    return default


def _read(obj, names, default=None):
    if obj is None:
        return default
    for name in names:
        try:
            if isinstance(obj, Mapping) and name in obj:
                value = obj[name]
            elif hasattr(obj, name):
                value = getattr(obj, name)
            else:
                continue
            if callable(value):
                continue
            if value is not None:
                return value
        except Exception:
            continue
    return default


def _walk_candidates(obj, max_depth=2):
    """Yield a small, safe object graph for score/state field discovery."""
    seen = set()
    queue = deque([(obj, 0)])
    child_names = (
        "state", "game_state", "observation", "result", "info", "metadata",
        "response", "frame", "board", "env", "game", "run",
    )
    while queue:
        cur, depth = queue.popleft()
        if cur is None or id(cur) in seen:
            continue
        seen.add(id(cur))
        yield cur
        if depth >= max_depth:
            continue
        for name in child_names:
            nxt = _read(cur, (name,), None)
            if nxt is not None and not isinstance(nxt, (str, bytes, int, float, bool)):
                queue.append((nxt, depth + 1))


def _find_number(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _num(_read(candidate, names, None), None)
            if value is not None:
                return value
    return default


def _find_bool(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _read(candidate, names, None)
            if isinstance(value, bool):
                return value
            if isinstance(value, (int, float)) and value in (0, 1):
                return bool(value)
            if isinstance(value, str):
                v = value.strip().lower()
                if v in {"true", "yes", "won", "lost", "done", "terminal", "game_over"}:
                    return True
                if v in {"false", "no", "playing", "active", "running"}:
                    return False
    return default


def _extract_game_id(*objs):
    names = ("game_id", "env_name", "environment_id", "game_name", "name")
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            value = _read(candidate, names, None)
            if value is not None:
                text = str(value).strip()
                if text:
                    return text
    return "unknown"


def _extract_action(args, kwargs):
    for key in ("action", "action_name", "action_spec", "move", "command"):
        if key in kwargs:
            return str(kwargs[key])
    for value in args:
        if value is None:
            continue
        text = str(value)
        if text and len(text) <= 300:
            return text
    return "unknown"


def _visual_payload(*objs):
    """Return the first likely 2-D/3-D visual state payload without mutating it."""
    field_names = (
        "board", "grid", "frame", "image", "observation", "pixels",
        "screen", "state_matrix", "board_state",
    )
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=2):
            for name in field_names:
                value = _read(candidate, (name,), None)
                if value is None:
                    continue
                try:
                    if hasattr(value, "tolist"):
                        value = value.tolist()
                    # Require a matrix-like payload; text observations are not
                    # useful for the deterministic HUD-band comparison.
                    if (
                        isinstance(value, (list, tuple))
                        and len(value) >= 3
                        and isinstance(value[0], (list, tuple))
                    ):
                        return value
                except Exception:
                    continue
    return None


def _hash_payload(value):
    if value is None:
        return None
    try:
        payload = json.dumps(
            value,
            sort_keys=True,
            default=str,
            separators=(",", ":"),
        )
    except Exception:
        payload = repr(value)
    if not payload or len(payload) <= 4:
        return None
    return hashlib.sha1(
        payload[:2_000_000].encode("utf-8", "replace")
    ).hexdigest()[:16]


def _core_visual_payload(value):
    """Remove only thin outer HUD/moves bands; retain almost the entire board.

    This is deliberately conservative. The no-impact detector fires only when
    the full frame changes while this core stays identical and there is no
    score/reward/level progress. It therefore cannot manufacture positive
    evidence; it only discounts likely cosmetic/HUD-only changes.
    """
    if value is None or not isinstance(value, (list, tuple)) or len(value) < 8:
        return value
    rows = list(value)
    width = min(
        (len(row) for row in rows if isinstance(row, (list, tuple))),
        default=0,
    )
    if width < 8:
        return value

    # Trim 6.25% from each edge, capped so at least 6x6 content remains.
    trim_y = min(max(1, len(rows) // 16), max(1, (len(rows) - 6) // 2))
    trim_x = min(max(1, width // 16), max(1, (width - 6) // 2))
    core = []
    for row in rows[trim_y:len(rows) - trim_y]:
        if isinstance(row, (list, tuple)):
            core.append(list(row)[trim_x:width - trim_x])
    return core or value


def _stable_signature(*objs):
    return _hash_payload(_visual_payload(*objs))


def _core_signature(*objs):
    visual = _visual_payload(*objs)
    return _hash_payload(_core_visual_payload(visual))

def _snapshot(api, result=None):
    objs = tuple(x for x in (result, api) if x is not None)
    score = _find_number(objs, ("score", "current_score", "total_score", "game_score", "final_score"), None)
    levels = _find_number(objs, ("levels_completed", "level_completed_count", "completed_levels", "level"), None)
    reward = _find_number((result,), ("reward", "score_delta", "delta_reward"), None)
    board_changed = _find_bool((result,), ("board_changed", "state_changed", "frame_changed", "changed"), None)
    level_completed = _find_bool((result,), ("level_completed", "completed_level", "level_won"), None)
    game_over = _find_bool(objs, ("game_over", "done", "terminal", "is_done", "finished"), None)
    won = _find_bool(objs, ("won", "is_won", "victory"), None)
    lost = _find_bool(objs, ("lost", "is_lost", "defeat"), None)
    signature = _stable_signature(result, api)
    core_signature = _core_signature(result, api)
    return {
        "score": score,
        "levels": int(levels) if levels is not None else None,
        "reward": reward,
        "board_changed": board_changed,
        "level_completed": level_completed,
        "game_over": game_over,
        "won": won,
        "lost": lost,
        "signature": signature,
        "core_signature": core_signature,
    }


@dataclass
class DWEGameState:
    game_id: str
    move: int = 0
    last_score: float = 0.0
    last_levels: int = 0
    game_weight: float = 0.0
    strategy_weight: float = 0.0
    combined_weight: float = 0.0
    progress_velocity: float = 0.0
    stall_streak: int = 0
    no_progress_streak: int = 0
    repeat_streak: int = 0
    no_impact_streak: int = 0
    success_protect_until: int = 0
    live_budget: int = 160
    decision: str = "EXPLORE"
    reason: str = "initial observation"
    last_signature: str | None = None
    last_core_signature: str | None = None
    seen_signatures: deque = field(default_factory=lambda: deque(maxlen=96))
    last_terms: dict = field(default_factory=dict)


class DifferenceWeightedExploitation:
    def __init__(self):
        self._states = {}
        self._lock = threading.RLock()

    def state(self, game_id):
        key = str(game_id or "unknown")
        with self._lock:
            if key not in self._states:
                self._states[key] = DWEGameState(game_id=key)
            return self._states[key]

    @staticmethod
    def _budget(weight):
        for threshold, budget in DWE_BUDGET_TIERS:
            if weight >= threshold:
                return int(min(LS20_MAX_MOVES, budget))
        return int(LS20_MAX_MOVES)

    def pre(self, game_id, action):
        with self._lock:
            st = self.state(game_id)
            print(
                "DWE PRE "
                f"game={st.game_id} move={st.move + 1:03d} action={action} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} mode={st.decision} "
                f"live_budget={st.live_budget}/{LS20_MAX_MOVES} "
                f"stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"protect_until={st.success_protect_until} reason={st.reason}",
                flush=True,
            )
            return {
                "move": st.move + 1,
                "score": st.last_score,
                "levels": st.last_levels,
                "signature": st.last_signature,
                "core_signature": st.last_core_signature,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "decision": st.decision,
            }

    def post(self, game_id, action, before, after):
        with self._lock:
            st = self.state(game_id)
            st.move += 1

            before_score = _num(before.get("score"), st.last_score)
            if before_score is None:
                before_score = st.last_score
            after_score = _num(after.get("score"), None)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = before_score + reward
            score_delta = after_score - before_score

            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            level_event = bool(after.get("level_completed"))
            if after_levels is None:
                after_levels = before_levels + (1 if level_event else 0)
            level_delta = max(0, int(after_levels) - int(before_levels))
            level_event = bool(level_event or level_delta > 0)

            sig = after.get("signature")
            prev_sig = st.last_signature
            seen_before = set(st.seen_signatures)
            novel = bool(sig and sig not in seen_before)
            repeated = bool(sig and (sig == prev_sig or sig in seen_before))
            board_changed = after.get("board_changed")
            if board_changed is None and sig and prev_sig:
                board_changed = sig != prev_sig
            if board_changed is None:
                board_changed = bool(score_delta != 0 or level_event or reward != 0)

            core_sig = after.get("core_signature")
            prev_core_sig = st.last_core_signature
            core_changed = None
            if core_sig and prev_core_sig:
                core_changed = core_sig != prev_core_sig

            positive_score = score_delta > 1e-9
            positive_reward = reward > 1e-9
            meaningful_progress = bool(level_event or positive_score or positive_reward)

            # NO_IMPACT = apparent frame/board activity confined to the thin outer
            # band, with no verified reward/score/level progress.
            no_impact = bool(
                board_changed
                and core_changed is False
                and not meaningful_progress
            )
            effective_board_changed = bool(board_changed and not no_impact)
            effective_novel = bool(novel and not no_impact)
            state_activity = bool(effective_board_changed or effective_novel)
            loop_signal = bool(repeated and not meaningful_progress)

            if no_impact:
                st.no_impact_streak += 1
            else:
                st.no_impact_streak = 0

            if meaningful_progress:
                st.no_progress_streak = 0
            else:
                st.no_progress_streak += 1

            if meaningful_progress or state_activity:
                st.stall_streak = 0
            else:
                st.stall_streak += 1

            if loop_signal:
                st.repeat_streak += 1
            else:
                st.repeat_streak = 0

            # Progress value is deliberately conservative: visual novelty alone is useful
            # information, but weaker than verified score/reward/level progress.
            progress_value = 0.0
            if level_event:
                progress_value += 1.00
            if positive_score:
                progress_value += min(0.65, 0.20 + abs(score_delta))
            if positive_reward:
                progress_value += min(0.40, 0.10 + abs(reward))
            if effective_board_changed:
                progress_value += 0.12
            if effective_novel:
                progress_value += 0.10
            if loop_signal:
                progress_value -= 0.30
            if no_impact:
                progress_value -= 0.25
            if not meaningful_progress and not state_activity:
                progress_value -= 0.15
            progress_value = _clip(progress_value, -1.0, 1.0)
            st.progress_velocity = _clip(0.70 * st.progress_velocity + 0.30 * progress_value, -1.0, 1.0)

            # Causal confidence is tied only to observed current-game consequences.
            causal_confidence = 0.0
            if level_event:
                causal_confidence = 1.0
            elif positive_score:
                causal_confidence = 0.85
            elif positive_reward:
                causal_confidence = 0.70
            elif effective_board_changed and effective_novel:
                causal_confidence = 0.35
            elif effective_board_changed:
                causal_confidence = 0.20

            normalized_score = _clip(after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 2.0)
            target_proximity = 1.0 if after_score >= TARGET_MIN_GAME_SCORE else _clip(
                after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 1.0
            )
            # A stagnant near-target game should not monopolize budget, so proximity is
            # gated by recent causal progress velocity.
            target_activity_gate = 0.20 + 0.80 * max(0.0, st.progress_velocity)
            target_signal = target_proximity * target_activity_gate
            stall_ratio = _clip(st.stall_streak / max(MAX_STALL_ACTIONS, 1), 0.0, 1.0)
            no_progress_ratio = _clip(st.no_progress_streak / max(MAX_NO_PROGRESS_ACTIONS, 1), 0.0, 1.0)
            repeat_ratio = _clip(st.repeat_streak / 4.0, 0.0, 1.0)
            no_impact_ratio = _clip(
                st.no_impact_streak / max(NO_IMPACT_STREAK_FOR_STOP, 1),
                0.0,
                1.0,
            )
            terminal_loss = bool(after.get("lost") or (after.get("game_over") and not after.get("won")))

            terms = {
                "score": EXPLOIT_WEIGHTS["score"] * normalized_score,
                "level_complete": EXPLOIT_WEIGHTS["level_complete"] * (1.0 if level_event else 0.0),
                "progress_velocity": EXPLOIT_WEIGHTS["progress_velocity"] * st.progress_velocity,
                "novel_state": EXPLOIT_WEIGHTS["novel_state"] * (1.0 if effective_novel else 0.0),
                "causal_confidence": EXPLOIT_WEIGHTS["causal_confidence"] * causal_confidence,
                "target_proximity": EXPLOIT_WEIGHTS["target_proximity"] * target_signal,
                "stall": EXPLOIT_WEIGHTS["stall"] * stall_ratio,
                "repeat_loop": EXPLOIT_WEIGHTS["repeat_loop"] * repeat_ratio,
                "no_progress": EXPLOIT_WEIGHTS["no_progress"] * no_progress_ratio,
                "no_impact": EXPLOIT_WEIGHTS["no_impact"] * no_impact_ratio,
                "terminal_loss": EXPLOIT_WEIGHTS["terminal_loss"] * (1.0 if terminal_loss else 0.0),
            }

            game_signal = sum(terms.values())
            # Strategy weight emphasizes immediate causal evidence and punishes local
            # failure more strongly than game weight, allowing CHANGE_POLICY on good games.
            strategy_signal = (
                5.00 * (1.0 if level_event else 0.0)
                + 2.75 * st.progress_velocity
                + 2.00 * causal_confidence
                + 0.50 * (1.0 if effective_novel else 0.0)
                - 2.50 * stall_ratio
                - 4.00 * repeat_ratio
                - 3.25 * no_progress_ratio
                - 4.25 * no_impact_ratio
                - 4.50 * (1.0 if terminal_loss else 0.0)
            )

            st.game_weight = _clip(
                GAME_WEIGHT_DECAY * st.game_weight + game_signal,
                -GAME_WEIGHT_LIMIT,
                GAME_WEIGHT_LIMIT,
            )
            st.strategy_weight = _clip(
                STRATEGY_WEIGHT_DECAY * st.strategy_weight + strategy_signal,
                -STRATEGY_WEIGHT_LIMIT,
                STRATEGY_WEIGHT_LIMIT,
            )
            st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight

            if level_event or positive_score or positive_reward:
                st.success_protect_until = max(
                    st.success_protect_until,
                    st.move + SUCCESS_PROTECT_ACTIONS,
                )

            st.live_budget = self._budget(st.combined_weight)
            if st.move <= st.success_protect_until:
                st.live_budget = max(st.live_budget, min(LS20_MAX_MOVES, st.success_protect_until + 12))

            protected = st.move <= st.success_protect_until
            if after.get("game_over"):
                st.decision = "TERMINAL"
                st.reason = "environment reported terminal state"
            elif st.move < MIN_OBSERVATION_ACTIONS:
                st.decision = "EXPLORE"
                st.reason = "minimum observation window"
            elif level_event and st.combined_weight >= 3.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "verified level completion"
            elif meaningful_progress and st.combined_weight >= 1.0:
                st.decision = "EXPLOIT"
                st.reason = "verified current-game progress"
            elif protected:
                st.decision = "EXPLOIT_PROTECTED"
                st.reason = "recent success protected exploit window"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_STOP and st.game_weight < 1.0:
                st.decision = "STOP_LOSS"
                st.reason = "repeated HUD/band-only no-impact actions with low game value"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact actions; abandon current local strategy"
            elif st.no_progress_streak >= MAX_NO_PROGRESS_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "promising game but current strategy has no progress"
                else:
                    st.decision = "STOP_LOSS"
                    st.reason = "sustained no progress with low game value"
            elif st.stall_streak >= MAX_STALL_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall detected in a still-promising game"
                elif st.combined_weight < -1.0:
                    st.decision = "STOP_LOSS"
                    st.reason = "hard stall plus negative exploit evidence"
                else:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall threshold reached"
            elif st.game_weight >= 1.0 and st.strategy_weight <= -1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "game weight high while strategy weight is low"
            elif st.combined_weight >= 6.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "very high combined exploit weight"
            elif st.combined_weight >= 3.0:
                st.decision = "EXPLOIT"
                st.reason = "high combined exploit weight"
            elif st.combined_weight >= 1.0:
                st.decision = "CAUTIOUS_EXPLOIT"
                st.reason = "positive combined exploit weight"
            elif st.combined_weight >= -1.0:
                st.decision = "BALANCED"
                st.reason = "mixed current-game evidence"
            else:
                st.decision = "EXPLORE"
                st.reason = "low exploit confidence; seek information"

            st.last_score = float(after_score)
            st.last_levels = int(after_levels)
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            if core_sig:
                st.last_core_signature = core_sig
            st.last_terms = {k: round(v, 6) for k, v in terms.items()}

            event = {
                "game_id": st.game_id,
                "move": st.move,
                "action": action,
                "before_score": before_score,
                "after_score": after_score,
                "score_delta": score_delta,
                "before_levels": before_levels,
                "after_levels": after_levels,
                "level_delta": level_delta,
                "reward": reward,
                "board_changed": bool(board_changed),
                "effective_board_changed": bool(effective_board_changed),
                "core_changed": core_changed,
                "no_impact": bool(no_impact),
                "novel_state": novel,
                "effective_novel_state": effective_novel,
                "loop_signal": loop_signal,
                "progress_value": progress_value,
                "progress_velocity": st.progress_velocity,
                "causal_confidence": causal_confidence,
                "target_proximity": target_proximity,
                "stall_streak": st.stall_streak,
                "no_progress_streak": st.no_progress_streak,
                "repeat_streak": st.repeat_streak,
                "no_impact_streak": st.no_impact_streak,
                "terms": st.last_terms,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "combined_weight": st.combined_weight,
                "decision": st.decision,
                "reason": st.reason,
                "live_budget": st.live_budget,
                "hard_budget": LS20_MAX_MOVES,
                "success_protect_until": st.success_protect_until,
                "game_over": bool(after.get("game_over")),
                "won": bool(after.get("won")),
                "lost": bool(after.get("lost")),
            }
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")

            term_text = ",".join(f"{k}={v:+.2f}" for k, v in st.last_terms.items())
            print(
                "DWE POST "
                f"game={st.game_id} move={st.move:03d} action={action} "
                f"score={before_score:.6f}->{after_score:.6f} dscore={score_delta:+.6f} "
                f"levels={before_levels}->{after_levels} dlevel={level_delta:+d} reward={reward:+.4f} "
                f"changed={int(bool(board_changed))} effective_changed={int(bool(effective_board_changed))} "
                f"NO_IMPACT={int(bool(no_impact))} novel={int(novel)} loop={int(loop_signal)} "
                f"progress={progress_value:+.3f} velocity={st.progress_velocity:+.3f} "
                f"target={target_proximity:.3f} stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} next={st.decision} "
                f"budget={st.live_budget}/{LS20_MAX_MOVES} protect_until={st.success_protect_until} "
                f"terms=[{term_text}] reason={st.reason}",
                flush=True,
            )
            return event

    def should_stop(self, game_id):
        with self._lock:
            st = self.state(game_id)
            if st.move >= LS20_MAX_MOVES:
                return True, "hard action ceiling"
            if st.move <= st.success_protect_until:
                return False, "success protected"
            if st.decision == "STOP_LOSS":
                return True, st.reason
            if (
                st.no_impact_streak >= NO_IMPACT_STREAK_FOR_STOP
                and st.game_weight < 1.0
            ):
                return True, "no-impact stop threshold"
            if (
                st.move >= st.live_budget
                and st.no_progress_streak >= MAX_STALL_ACTIONS
                and st.combined_weight < 1.0
            ):
                return True, "dynamic live budget exhausted without exploit evidence"
            return False, "continue"

    def summaries(self):
        with self._lock:
            return [
                {
                    "game_id": st.game_id,
                    "moves": st.move,
                    "score": st.last_score,
                    "levels": st.last_levels,
                    "game_weight": st.game_weight,
                    "strategy_weight": st.strategy_weight,
                    "combined_weight": st.combined_weight,
                    "decision": st.decision,
                    "reason": st.reason,
                    "live_budget": st.live_budget,
                    "stall_streak": st.stall_streak,
                    "no_progress_streak": st.no_progress_streak,
                    "repeat_streak": st.repeat_streak,
                    "no_impact_streak": st.no_impact_streak,
                "no_impact_streak": st.no_impact_streak,
                }
                for st in self._states.values()
            ]


DWE_ALLOCATOR = DifferenceWeightedExploitation()
_DWE_ACTION_DEPTH = contextvars.ContextVar("dwe_action_depth", default=0)
_DWE_PATCHED_ACTION_METHODS = []
_DWE_PATCHED_STOP_METHODS = []


def _method_action_score(name, method):
    score = 0
    lname = name.lower()
    if lname in {"step", "act", "action", "execute_action", "perform_action", "take_action", "play_action", "apply_action"}:
        score += 8
    if "action" in lname:
        score += 4
    if any(token in lname for token in ("step", "move", "act", "play")):
        score += 2
    try:
        sig = inspect.signature(method)
        params = {p.lower() for p in sig.parameters}
        if params.intersection({"action", "action_name", "action_spec", "move", "command"}):
            score += 8
    except Exception:
        pass
    try:
        source = inspect.getsource(method).lower()
        if "board_changed" in source or "level_completed" in source or ".step(" in source:
            score += 4
    except Exception:
        pass
    return score


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_gameapi_action_hook(game_apis):
    classes = []
    for api in game_apis:
        if api.__class__ not in classes:
            classes.append(api.__class__)
    installed = []
    for cls in classes:
        candidates = []
        for name in dir(cls):
            if name.startswith("_"):
                continue
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if not callable(method):
                continue
            score = _method_action_score(name, method)
            if score > 0:
                candidates.append((score, name))
        candidates.sort(reverse=True)
        if not candidates:
            raise RuntimeError(
                f"DWE could not identify a real action method on {cls.__module__}.{cls.__name__}; "
                "refusing to run without per-move exploit logging."
            )
        best_score, best_name = candidates[0]
        if best_score < 6:
            raise RuntimeError(
                f"DWE action-boundary confidence too low for {cls.__name__}: {candidates[:8]}"
            )
        if _wrap_action_method(cls, best_name):
            installed.append(f"{cls.__module__}.{cls.__name__}.{best_name}")
        print(
            f"DWE ACTION HOOK class={cls.__module__}.{cls.__name__} "
            f"method={best_name} confidence={best_score} candidates={candidates[:6]}",
            flush=True,
        )
    return installed


def _object_game_id(self, args, kwargs):
    return _extract_game_id(self, *args, *kwargs.values())


def _wrap_should_stop(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_stop_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            original_result = await original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result
    else:
        def wrapped(self, *args, **kwargs):
            original_result = original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_stop_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_STOP_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_should_stop_hooks(game_apis):
    classes = {api.__class__ for api in game_apis}
    # The stop predicate can live on GameAPI, a solver/session class, or a run class.
    for module_name, module in list(sys.modules.items()):
        if not module or not (
            module_name.startswith("taaf") or module_name.startswith("inference")
        ):
            continue
        try:
            values = list(vars(module).values())
        except Exception:
            continue
        for obj in values:
            if inspect.isclass(obj):
                classes.add(obj)

    installed = []
    for cls in classes:
        for name in ("should_stop", "_should_stop"):
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if callable(method) and _wrap_should_stop(cls, name):
                full = f"{cls.__module__}.{cls.__name__}.{name}"
                installed.append(full)
                print(f"DWE STOP HOOK {full}", flush=True)
    if not installed:
        print(
            "DWE STOP HOOK WARNING: no should_stop predicate found; hard ceiling remains active. "
            "Per-move DWE logging and policy weighting are still active.",
            flush=True,
        )
    return installed


def _install_dwe_runtime_hooks(game_apis):
    action_hooks = _install_gameapi_action_hook(game_apis)
    stop_hooks = _install_should_stop_hooks(game_apis)
    if not action_hooks and not _DWE_PATCHED_ACTION_METHODS:
        raise RuntimeError("DWE requires an action-boundary hook; none was installed.")
    print(
        "DWE RUNTIME ACTIVE "
        f"action_hooks={len(action_hooks) or len(_DWE_PATCHED_ACTION_METHODS)} "
        f"stop_hooks={len(stop_hooks) or len(_DWE_PATCHED_STOP_METHODS)} "
        "log_every_move=True",
        flush=True,
    )


print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("DWE ACTIVE: separate game and strategy exploit weights", flush=True)
print("DWE LOGGING: PRE + POST for every real environment action", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME MEMORY: DISABLED", flush=True)


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(os.environ["ARC_AGI3_ENVIRONMENTS_DIR"])
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# Install deterministic per-move exploit logging and stop-loss hooks only after
# the actual GameAPI class(es) have been constructed. The notebook refuses to
# start the benchmark if it cannot identify the action boundary confidently.
_install_dwe_runtime_hooks(game_apis)

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
# Keep the per-game runtime aligned with the ls20 move budget target.
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "ls20_max_moves": LS20_MAX_MOVES,
    "control_seed": CONTROL_SEED,
    "analyzer_model": os.environ.get("INFERENCE_ANALYZER_MODEL"),
    "qwen_model_dir": str(QWEN_MODEL_DIR),
    "resolved_model_dataset": RESOLVED_MODEL_DATASET_SOURCE,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
    "frame_mode": os.environ.get("ARC3_FRAME_MODE"),
    "state_graph": os.environ.get("ARC3_STATE_GRAPH"),
    "no_impact_weight": EXPLOIT_WEIGHTS["no_impact"],
    "no_impact_policy_change": NO_IMPACT_STREAK_FOR_POLICY_CHANGE,
    "no_impact_stop": NO_IMPACT_STREAK_FOR_STOP,
    "schema": "adldb.arc3.dwe.qwen38.seed20260819.v3",
    "dwe_enabled": True,
    "dwe_log_every_move": True,
    "dwe_weights": EXPLOIT_WEIGHTS,
    "dwe_budget_tiers": [list(x) for x in DWE_BUDGET_TIERS],
    "success_protect_actions": SUCCESS_PROTECT_ACTIONS,
    "min_observation_actions": MIN_OBSERVATION_ACTIONS,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
    "target_score_games": TARGET_SCORE_GAMES,
    "target_min_game_score": TARGET_MIN_GAME_SCORE,
    "max_stall_actions": MAX_STALL_ACTIONS,
    "max_no_progress_actions": MAX_NO_PROGRESS_ACTIONS,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "QWEN38 ADLDB-DWE RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f} "
    f"hard_action_ceiling={LS20_MAX_MOVES} dwe=on log_every_move=on "
    f"seed={CONTROL_SEED} frame=full model={os.environ.get('INFERENCE_ANALYZER_MODEL')}",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "ADLDB-DWE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


## 10. Final ADLDB/DWE run summary

Summarize scored outcomes and the final exploit state of every game. The output keeps the score metrics needed for the next ADL comparison and writes a compact per-game DWE summary artifact.


In [ ]:
# === FINAL ADLDB / DWE SUMMARY ===
import statistics

runs = list(getattr(bm, "game_runs", []) or [])
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]
positive = sum(score > 0 for score in scores)
at_target = sum(score >= TARGET_MIN_GAME_SCORE for score in scores)

print(
    "QWEN38 ADLDB SUMMARY "
    f"games={len(runs)} "
    f"mean_score={(sum(scores) / len(scores) if scores else 0.0):.6f} "
    f"score_sum={sum(scores):.6f} "
    f"positive_games={positive} "
    f"target_games={at_target}/{TARGET_SCORE_GAMES} "
    f"levels={sum(levels)} "
    f"actions={sum(actions)} "
    f"seed={CONTROL_SEED} frame={os.environ.get('ARC3_FRAME_MODE')} "
    f"model={os.environ.get('INFERENCE_ANALYZER_MODEL')}",
    flush=True,
)

summaries = DWE_ALLOCATOR.summaries()
with DWE_SUMMARY_LOG.open("w", encoding="utf-8") as fh:
    for item in sorted(summaries, key=lambda x: x["game_id"]):
        fh.write(json.dumps(item, sort_keys=True) + "\n")
        print(
            "DWE GAME SUMMARY "
            f"game={item['game_id']} moves={item['moves']} score={item['score']:.6f} "
            f"levels={item['levels']} gameW={item['game_weight']:+.3f} "
            f"strategyW={item['strategy_weight']:+.3f} combined={item['combined_weight']:+.3f} "
            f"decision={item['decision']} live_budget={item['live_budget']} "
            f"stall={item['stall_streak']} no_progress={item['no_progress_streak']} "
            f"repeat={item['repeat_streak']} "
            f"no_impact={item.get('no_impact_streak', 0)} reason={item['reason']}",
            flush=True,
        )

print(f"DWE MOVE LOG: {DWE_MOVE_LOG}", flush=True)
print(f"DWE SUMMARY LOG: {DWE_SUMMARY_LOG}", flush=True)


## 11. Per-move exploit/ADL audit

Verify that the deterministic DWE action-boundary logger produced one unique post-action exploit record for every recorded game action. In local validation mode, incomplete coverage is a hard failure. In official competition reruns it is surfaced prominently without destroying an otherwise valid submission artifact.


In [ ]:
# === PER-MOVE DWE / POST-MOVE ADL AUDIT ===
from collections import Counter

records = []
if DWE_MOVE_LOG.exists():
    for raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("DWE AUDIT malformed line:", raw[:240], flush=True)

unique_move_keys = {
    (str(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
recorded_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
logged_moves = len(unique_move_keys)
coverage = logged_moves / recorded_actions if recorded_actions else 1.0

modes = Counter(str(item.get("decision", "unknown")) for item in records)
stop_loss = modes.get("STOP_LOSS", 0)
no_impact_moves = sum(1 for item in records if item.get("no_impact"))
change_policy = modes.get("CHANGE_POLICY", 0)
exploit_moves = sum(
    count for mode, count in modes.items()
    if "EXPLOIT" in mode
)

print(
    "DWE MOVE AUDIT "
    f"recorded_actions={recorded_actions} "
    f"unique_logged_moves={logged_moves} "
    f"coverage={coverage:.3f} "
    f"exploit_updates={exploit_moves} "
    f"change_policy_updates={change_policy} "
    f"stop_loss_updates={stop_loss} "
    f"no_impact_moves={no_impact_moves} "
    f"modes={dict(sorted(modes.items()))}",
    flush=True,
)

# Per-game coverage makes any missing trace immediately visible in notebook logs.
logged_by_game = Counter(str(item.get("game_id", "unknown")) for item in records)
for run in getattr(bm, "game_runs", []) or []:
    gid = _game_key(run.game_id)
    # Match either exact full id or normalized game key.
    logged = sum(
        count for key, count in logged_by_game.items()
        if _game_key(key) == gid
    )
    expected = _run_actions(run)
    game_cov = logged / expected if expected else 1.0
    print(
        "DWE GAME AUDIT "
        f"game={gid} actions={expected} logged={logged} coverage={game_cov:.3f}",
        flush=True,
    )

if coverage < 0.999999:
    message = (
        "DWE per-move exploit logging coverage is incomplete: "
        f"{logged_moves}/{recorded_actions} ({coverage:.3%})."
    )
    if DWE_STRICT_LOG_COVERAGE and not TRUE_SUBMISSION:
        raise RuntimeError(message)
    print("WARNING:", message, flush=True)
else:
    print("DWE AUDIT PASS: exploit logic logged for every recorded move", flush=True)
